<a href="https://colab.research.google.com/github/fonglieu/text-analytics-assignment5/blob/main/Assignment_5_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Fong Lieu
**Option:** B — Job Fit Analyzer  
**API Path:** [Paid / Free]  

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [1]:
# ── Install packages (uncomment as needed) ──
!pip install openai pypdf pandas chromadb sentence-transformers -q

from google.colab import drive
drive.mount('/content/drive')

# ── Your imports below ──
import os
import re
import pandas as pd
from collections import Counter
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from openai import OpenAI
from google.colab import userdata

# ── API key ──
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ── Paths ──
BASE_DIR = "/content/drive/MyDrive/data"

print("Imports successful.")
print(f"Files: {os.listdir(BASE_DIR)}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [2]:
# ── Load JD metadata ──
metadata_df = pd.read_csv(os.path.join(BASE_DIR, "jd_metadata.csv"))
print(f"JDs loaded: {len(metadata_df)}\n")
display(metadata_df)

JDs loaded: 10



,filename,company,title,source_url,date_collected
0,jd_01_amend_sales_analyst.txt,AMEND Consulting,Sales Analyst – Co-op/Internship,https://job-boards.greenhouse.io/amendconsulti...,2026-05-09
1,jd_02_att_strategy_analyst.txt,AT&T,Strategy Analyst – Wireline Transformation & S...,https://www.att.jobs/job/-/strategy-analyst/11...,2026-05-09
2,jd_03_spotify_data_analyst.txt,Spotify,Data Analyst II – Performance Optimization Squad,https://jobs.lever.co/spotify/d95e1989-9a1a-48...,2026-05-09
3,jd_04_delta_data_analyst.txt,Delta Air Lines,Senior Analyst – Customer Data Solutions,https://delta.avature.net/en_US/careers/JobDet...,2026-05-09
4,jd_05_northrop_grumman_data_analyst.txt,Northrop Grumman,Data Insight Analyst,https://jobs.northropgrumman.com/careers/job/1...,2026-05-09
5,jd_06_transformlabs_business_analyst.txt,Transform Labs,Entry-Level Business Analyst,https://awhnet.bamboohr.com/careers/76,2026-05-09
6,jd_07_qualifiedhealth_product_intern.txt,Qualified Health,Product Intern,https://lmu.joinhandshake.com/jobs/10994147,2026-05-09
7,jd_08_capgemini_junior_ba.txt,Capgemini America Inc.,Junior Business Analyst,https://lmu.joinhandshake.com/job-search/10982553,2026-05-09
8,jd_09_bayview_data_ops_analyst.txt,Bayview Asset Management,Data Operations Analyst,https://lmu.joinhandshake.com/job-search/10974135,2026-05-09
9,jd_10_goldmansachs_data_office_analyst.txt,Goldman Sachs,Asset & Wealth Management Data Office Analyst,https://www.linkedin.com/jobs/view/4375782060/,2026-05-09


In [3]:
# ── Load JD documents ──
def load_text_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

jd_docs = {}
for _, row in metadata_df.iterrows():
    filepath = os.path.join(BASE_DIR, row["filename"])
    jd_docs[row["filename"]] = {
        "text":       load_text_file(filepath),
        "company":    row["company"],
        "title":      row["title"],
        "source_url": row["source_url"]
    }

print(f"JD documents loaded: {len(jd_docs)}")

JD documents loaded: 10


In [4]:
# ── Load Resume ──
def load_pdf(filepath):
    reader = PdfReader(filepath)
    return "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])

resume_text = load_pdf(os.path.join(BASE_DIR, "Fong_Lieu_Resume.pdf"))
print(f"Resume loaded. Length: {len(resume_text)} chars")

Resume loaded. Length: 3438 chars


In [5]:
# ── Preview sample content ──
sample_key = list(jd_docs.keys())[0]
print(f"=== Sample JD: {sample_key} ===")
print(jd_docs[sample_key]["text"][:500])
print("\n=== Resume (first 500 chars) ===")
print(resume_text[:500])

=== Sample JD: jd_01_amend_sales_analyst.txt ===
Company: AMEND Consulting
Title: Sales Analyst – Co-op/Internship
Location: Cincinnati, OH
Source: https://job-boards.greenhouse.io/amendconsulting/jobs/4005828009
Date Collected: 2026-05-09

--- JOB DESCRIPTION ---

About AMEND:
AMEND is a management consulting firm based in Cincinnati, OH with areas of focus in operations, analytics, and technology. We are focused on strengthening the people, processes, and systems in organizations to generate a holistic transformation. Our three-tiered approa

=== Resume (first 500 chars) ===
Fong  Lieu
 
Los  Angeles,  CA   |   (720)  921-4111   |   fonglieu8@gmail.com   
TECHNICAL  SKILLS  Programming  &  Statistical  Languages:  SQL,  Python,  R  Libraries  &  Frameworks:  pandas,  NumPy,  matplotlib,  scikit-learn,  dplyr,  tidyr  Data  Analysis  &  Visualization:  Tableau,  Power  BI,  Excel  Data  Management:  Data  Extraction  &  Transformation,  Databases,  Salesforce  EDUCATION  Loyola  Marym

---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [6]:
# ── Strategy 1 ──
def chunk_text(text, chunk_size=600, overlap=75):
    """Strategy 1: fixed-size, may split mid-sentence."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]

# Run Strategy 1 on all documents
chunks_1 = []
for filename, doc in jd_docs.items():
    chunks_1.extend(chunk_text(doc["text"], chunk_size=600, overlap=75))
chunks_1.extend(chunk_text(resume_text, chunk_size=600, overlap=75))

print("Strategy 1: Fixed-Size Chunking")
print(f"  Total chunks:      {len(chunks_1)}")
print(f"  Avg length (chars): {round(sum(len(c) for c in chunks_1) / len(chunks_1))}")
print(f"  Min chunk:          {min(len(c) for c in chunks_1)}")
print(f"  Max chunk:          {max(len(c) for c in chunks_1)}")
print(f"\nSample chunk:")
print(chunks_1[3])

Strategy 1: Fixed-Size Chunking
  Total chunks:      74
  Avg length (chars): 544
  Min chunk:          33
  Max chunk:          600

Sample chunk:
aintenance of sales activity data in Salesforce and other key systems
- Prepare weekly sales reports and analyze data through Excel manipulation to track key performance indicators (KPIs)
- Generate and elevate insights on ways to improve sales support processes, systems, and data
- Support the creation and organization of sales materials, inclusive of presentations, one-pagers, and more
- Own and execute elements of event planning and delivery within the sales function
- Take on additional responsibilities as needed to support the broader sales and business development efforts

Qualifications


In [7]:
# ── Strategy 2 ──
def chunk_text_by_sentences(text, chunk_size=600, overlap=75):
    """Strategy 2: respects sentence boundaries."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            words = current_chunk.split()
            overlap_text = " ".join(words[-10:]) if len(words) > 10 else current_chunk
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += (" " if current_chunk else "") + sentence
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

# Run Strategy 2 on all documents, store with metadata
all_chunks = []
for filename, doc in jd_docs.items():
    doc_chunks = chunk_text_by_sentences(doc["text"], chunk_size=600, overlap=75)
    for chunk in doc_chunks:
        all_chunks.append({
            "text":    chunk,
            "source":  filename,
            "company": doc["company"]
        })

resume_chunks = chunk_text_by_sentences(resume_text, chunk_size=600, overlap=75)
for chunk in resume_chunks:
    all_chunks.append({
        "text":    chunk,
        "source":  "Fong_Lieu_Resume.pdf",
        "company": "resume"
    })

chunks_2 = [c["text"] for c in all_chunks]

print("Strategy 2: Sentence-Aware Chunking")
print(f"  Total chunks:       {len(chunks_2)}")
print(f"  Avg length (chars): {round(sum(len(c) for c in chunks_2) / len(chunks_2))}")
print(f"  Min chunk:          {min(len(c) for c in chunks_2)}")
print(f"  Max chunk:          {max(len(c) for c in chunks_2)}")
print(f"\nChunks per source:")
for source, count in Counter(c["source"] for c in all_chunks).items():
    print(f"  {source}: {count}")
print(f"\nSample chunk:")
print(chunks_2[3])

Strategy 2: Sentence-Aware Chunking
  Total chunks:       44
  Avg length (chars): 866
  Min chunk:          220
  Max chunk:          2750

Chunks per source:
  jd_01_amend_sales_analyst.txt: 4
  jd_02_att_strategy_analyst.txt: 3
  jd_03_spotify_data_analyst.txt: 4
  jd_04_delta_data_analyst.txt: 3
  jd_05_northrop_grumman_data_analyst.txt: 4
  jd_06_transformlabs_business_analyst.txt: 3
  jd_07_qualifiedhealth_product_intern.txt: 3
  jd_08_capgemini_junior_ba.txt: 4
  jd_09_bayview_data_ops_analyst.txt: 4
  jd_10_goldmansachs_data_office_analyst.txt: 6
  Fong_Lieu_Resume.pdf: 6

Sample chunk:
program with curriculum to build a wholistic understanding of business. Responsibilities:
- Develop a strong understanding of AMEND's service offerings and approach to client engagement
- Research and identify potential sales leads and market trends to support an active and organized pipeline for the sales team, using tools such as ZoomInfo and Hunter
- Ensure accurate entry and ongoing maintena

In [8]:
# ── Compare strategies ──
comparison = pd.DataFrame({
    "Strategy": [
        "Strategy 1: Fixed-Size (600/75)",
        "Strategy 2: Sentence-Aware (600/75)"
    ],
    "Total Chunks": [len(chunks_1), len(chunks_2)],
    "Avg Length (chars)": [
        round(sum(len(c) for c in chunks_1) / len(chunks_1)),
        round(sum(len(c) for c in chunks_2) / len(chunks_2))
    ],
    "Min Chunk": [
        min(len(c) for c in chunks_1),
        min(len(c) for c in chunks_2)
    ],
    "Max Chunk": [
        max(len(c) for c in chunks_1),
        max(len(c) for c in chunks_2)
    ]
})
display(comparison)

print("\n=== Strategy 1 sample chunk ===")
print(chunks_1[3])
print("\n=== Strategy 2 sample chunk ===")
print(chunks_2[3])

,Strategy,Total Chunks,Avg Length (chars),Min Chunk,Max Chunk
0,Strategy 1: Fixed-Size (600/75),74,544,33,600
1,Strategy 2: Sentence-Aware (600/75),44,866,220,2750



=== Strategy 1 sample chunk ===
aintenance of sales activity data in Salesforce and other key systems
- Prepare weekly sales reports and analyze data through Excel manipulation to track key performance indicators (KPIs)
- Generate and elevate insights on ways to improve sales support processes, systems, and data
- Support the creation and organization of sales materials, inclusive of presentations, one-pagers, and more
- Own and execute elements of event planning and delivery within the sales function
- Take on additional responsibilities as needed to support the broader sales and business development efforts

Qualifications

=== Strategy 2 sample chunk ===
program with curriculum to build a wholistic understanding of business. Responsibilities:
- Develop a strong understanding of AMEND's service offerings and approach to client engagement
- Research and identify potential sales leads and market trends to support an active and organized pipeline for the sales team, using tools such as

### Chunking Decision

**Which strategy did you choose?**
Strategy 2: Sentence-Aware Chunking (600/75)

**Why?**
I tested two strategies. Strategy 1 split the text at a fixed character limit of
600, which produced 74 chunks but caused a lot of mid-sentence and even mid-word
breaks. The minimum chunk size came out to 33 characters, which tells me the splitter was regularly creating fragments too small to be significant. Looking at the sample output, chunk index 3 started with "aintenance of sales activity data in Salesforce" which is literally the middle of the word "maintenance." This kind of chunk would return a result that is not useful in a similarity search because it has no semantic context.

Strategy 2 split on sentence boundaries instead of character limits, which
produced 44 chunks with an average size of 866 characters and a minimum of 220.
Every chunk contains at least one complete thought. The sample output shows it
kept the entire Responsibilities section and the full Qualifications section of
the AMEND job description together as a single chunk. That is more of what I
wanted because when the system retrieves content for a query like "Python SQL
requirements," it should return a complete list of qualifications and not just a fragment of one bullet point.

The tradeoff is that Strategy 2 has a max chunk size of 2,750 characters compared to Strategy 1's hard cap of 600. But for this use case a larger chunk that preserves meaning is more useful to the LLM than a smaller chunk that breaks it in half. Strategy 2 also produced fewer chunks overall (44
vs 74), which keeps the vector store lean and makes retrieval faster.

**Final settings:** chunk_size=600, overlap=75

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [9]:
# ── Create embeddings and vector store ──
import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="job_fit_analyzer",
    metadata={"hnsw:space": "cosine"}
)

if collection.count() == 0:
    collection.add(
        documents=[c["text"]    for c in all_chunks],
        metadatas=[{"source": c["source"], "company": c["company"]} for c in all_chunks],
        ids=[f"chunk_{i}"       for i in range(len(all_chunks))]
    )

print(f"Vector store created with {collection.count()} chunks.")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 22.4MiB/s]


Vector store created with 44 chunks.


I chose ChromaDB because it is lightweight, runs in-memory without a server,
and supports cosine similarity via the hnsw:space parameter. It is needed in this case because we care about the angle between vectors (semantic similarity) rather than their absolute distance. The sentence-transformers all-MiniLM-L6-v2 model was chosen because it is fast, free, and specifically trained for semantic similarity tasks, making it a good for matching job description requirements against resume content.

In [10]:
# ── Verify: run a test similarity search ──
def search(query, k=3):
    """Search the vector store and return top-k results with sources and distances."""
    results = collection.query(
        query_texts=[query],
        n_results=k
    )
    docs      = results["documents"][0]
    sources   = [m["source"]  for m in results["metadatas"][0]]
    companies = [m["company"] for m in results["metadatas"][0]]
    distances = results["distances"][0]
    return list(zip(docs, sources, companies, distances))

test_query = "SQL Python data analysis dashboard experience required"
print(f'Query: "{test_query}"\n')

for i, (text, source, company, dist) in enumerate(search(test_query, k=4)):
    print(f"Result {i+1} [source: {company}] (distance: {dist:.3f}):")
    print(f"  {text[:200]}...")
    print()

Query: "SQL Python data analysis dashboard experience required"

Result 1 [source: Northrop Grumman] (distance: 0.326):
  to communicate complex findings in a simple and actionable way. Responsibilities:
- Develop and maintain Tableau dashboards and data visualizations to support operational and strategic decisions
- Use...

Result 2 [source: Delta Air Lines] (distance: 0.391):
  layer and help build scalable solutions that drive business decisions. Responsibilities:
- Analyze large datasets related to customer behavior, scheduling, performance, and operational metrics
- Devel...

Result 3 [source: Spotify] (distance: 0.472):
  and metrics that make performance inefficiencies visible across the platform. Responsibilities:
- Design and own datasets, pipelines, and metrics that expose performance inefficiencies across the plat...

Result 4 [source: Northrop Grumman] (distance: 0.507):
  and logistics and modernization for our customers around the globe. About the Role:
The Data Insight A

In [11]:
# ── Edge case testing ──

edge_cases = [
    # Standard query
    ("Standard query", "Python SQL data analysis dashboard"),
    # Very short query
    ("Very short query", "Python"),
    # Unrelated query
    ("Unrelated query", "cooking recipes and food preparation"),
    # Query that could match multiple JDs
    ("Ambiguous query - matches multiple JDs", "financial services data analyst SQL"),
]

for label, query in edge_cases:
    print("=" * 60)
    print(f"Test: {label}")
    print(f"Query: '{query}'")
    print("=" * 60)
    results = search(query, k=3)
    for i, (text, source, company, dist) in enumerate(results):
        print(f"  Result {i+1} [{company}] (distance: {dist:.3f})")
        print(f"  {text[:150]}...")
    print()

Test: Standard query
Query: 'Python SQL data analysis dashboard'
  Result 1 [Northrop Grumman] (distance: 0.402)
  to communicate complex findings in a simple and actionable way. Responsibilities:
- Develop and maintain Tableau dashboards and data visualizations to...
  Result 2 [Delta Air Lines] (distance: 0.462)
  layer and help build scalable solutions that drive business decisions. Responsibilities:
- Analyze large datasets related to customer behavior, schedu...
  Result 3 [Northrop Grumman] (distance: 0.561)
  and logistics and modernization for our customers around the globe. About the Role:
The Data Insight Analyst will support analytics initiatives across...

Test: Very short query
Query: 'Python'
  Result 1 [Northrop Grumman] (distance: 0.667)
  to communicate complex findings in a simple and actionable way. Responsibilities:
- Develop and maintain Tableau dashboards and data visualizations to...
  Result 2 [Delta Air Lines] (distance: 0.727)
  layer and help build scalable s

---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [12]:
# ── Initialize LLM ──
def call_llm(prompt, model="gpt-4o-mini", temperature=0):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [13]:
# Helper Function
def get_jd_text(filename):
    return jd_docs[filename]["text"]

target_jd = get_jd_text("jd_10_goldmansachs_data_office_analyst.txt")
print("Helper ready.")

Helper ready.


In [14]:
# ── Analysis 1: Skill Gap Report ──
def skill_gap_report(jd_text, resume_text):
    prompt = f"""
You are a career coach analyzing a candidate's resume against a job description.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Produce a structured Skill Gap Report with exactly three sections:

MATCHING SKILLS:
- List each skill or experience the candidate has that directly matches a JD requirement.
- Be specific, using evidence from the resume.

MISSING SKILLS:
- List each requirement from the JD that is not present in the resume.
- Be specific about what is missing.

RECOMMENDED ACTIONS:
- For each missing skill, provide one concrete, actionable step the candidate can take
  (e.g., specific course, certification, or tool to learn).

Use bullet points and be concise. Do not create skills not present in either document.
"""
    return call_llm(prompt)

result = skill_gap_report(target_jd, resume_text)
print("=== SKILL GAP REPORT: Goldman Sachs ===\n")
print(result)

=== SKILL GAP REPORT: Goldman Sachs ===

### SKILL GAP REPORT

#### MATCHING SKILLS:
- **Data Analysis Experience**: The candidate has experience as a Pricing Analyst where they performed data extraction and transformation, which aligns with the JD's requirement for data analysis and operations experience.
- **Technical Skills**: Proficiency in SQL and Python, as well as experience with Excel, Tableau, and Power BI, matches the JD's requirement for technical skills in data analysis and reporting.
- **KPI Monitoring**: Developed KPI dashboards in Tableau for a team, which supports the JD's responsibility of monitoring platform-specific KPIs to identify data quality issues.
- **Documentation Skills**: Experience in creating practice problems and grading exams indicates the ability to document processes and data flows, aligning with the JD's requirement for documenting data definitions and ownership.
- **Project Support**: The candidate's role in preparing rate proposals and improving win

In [15]:
# ── Analysis 2: Keyword Alignment ──
def keyword_alignment(jd_text, resume_text):
    prompt = f"""
You are a resume keyword analyst.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Step 1 — Extract the 15 most important keywords and phrases from the job description,
with focus on required skills, tools, technologies, and domain terms.

Step 2 — For each keyword, determine if it appears or closely relates to the words in the resume.
Mark each as: MATCH, PARTIAL MATCH, or MISSING.

Step 3 — Calculate: match rate = (MATCH + 0.5 * PARTIAL MATCH) / 15

Output format:

KEYWORD TABLE:
| Keyword | Status | Evidence from Resume |
|---------|--------|----------------------|
...

MATCH RATE: X%

SUMMARY: One sentence interpreting the match rate and what it means for this application.
"""
    return call_llm(prompt)

result = keyword_alignment(target_jd, resume_text)
print("=== KEYWORD ALIGNMENT: Goldman Sachs ===\n")
print(result)

=== KEYWORD ALIGNMENT: Goldman Sachs ===

### KEYWORD TABLE:
| Keyword                                      | Status        | Evidence from Resume                                                                 |
|----------------------------------------------|---------------|--------------------------------------------------------------------------------------|
| Data governance                              | MISSING       | No mention of data governance in the resume.                                        |
| Data quality                                 | PARTIAL MATCH | Mention of data-driven identification of performance gaps in the Professional Learning Facilitator role. |
| KPIs                                         | MATCH         | Developed KPI dashboards in Tableau for a team of 5.                                |
| Data validation                              | MISSING       | No specific mention of data validation in the resume.                               |
| Data rec

In [16]:
# ── Analysis 3: Fit Summary ──
def fit_summary_v1(jd_text, resume_text):
    prompt = f"""
You are a hiring consultant. Read this job description and resume
and write a short summary of how well the candidate fits the role.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}
"""
    return call_llm(prompt)

result_v1 = fit_summary_v1(target_jd, resume_text)
print("=== FIT SUMMARY: Goldman Sachs ===\n")
print(result_v1)

=== FIT SUMMARY: Goldman Sachs ===

**Candidate Fit Summary:**

Fong Lieu presents a strong fit for the Analyst position within Goldman Sachs' Asset & Wealth Management, External Investing Group, Data Office. With a Master's degree in Business Analytics and a Bachelor's in Economics with a minor in Data Science, Fong possesses the educational background that aligns well with the role's requirements. 

Fong has relevant experience in data analysis and operations, demonstrated through their current role as a Pricing Analyst, where they performed data extraction and transformation, developed KPI dashboards in Tableau, and supported initiatives aimed at improving operational efficiency. This experience directly correlates with the responsibilities of monitoring KPIs, preparing reports, and ensuring data quality as outlined in the job description.

Additionally, Fong's technical skills in SQL, Python, and data visualization tools like Tableau and Power BI are well-suited for the analytical 

In [17]:
# ── Fit Summary: Iteration 1 ──
def fit_summary_v2(jd_text, resume_text):
    prompt = f"""
You are a hiring consultant writing a brief fit assessment.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Write a 3-4 sentence summary assessing how well this candidate fits the role.
Begin with an overall fit level: Strong, Moderate, or Weak.
Keep your response to 3-4 sentences only.
"""
    return call_llm(prompt)

result_v2 = fit_summary_v2(target_jd, resume_text)
print("=== FIT SUMMARY v2: Goldman Sachs ===\n")
print(result_v2)

=== FIT SUMMARY v2: Goldman Sachs ===

**Fit Level: Strong.** Fong Lieu possesses a solid educational background in Economics and Business Analytics, complemented by relevant technical skills in SQL, Python, and data visualization tools like Tableau and Power BI. With experience in data extraction, transformation, and KPI dashboard development, he aligns well with the responsibilities of supporting data governance and quality initiatives at Goldman Sachs. Additionally, his proactive approach to leveraging data for operational improvements and his interest in automation and AI solutions further enhance his suitability for the Analyst role in the XIG Data Office.


In [18]:
# ── Fit Summary: Iteration 2 ──
def fit_summary_v3(jd_text, resume_text):
    prompt = f"""
You are a hiring consultant writing a brief fit assessment.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Write a 3-4 sentence narrative Fit Summary assessing how well this candidate fits the role.

- Sentence 1: State overall fit level (Strong / Moderate / Weak) and the primary reason.
- Sentence 2: Highlight the candidate's single strongest relevant qualification for this role.
- Sentence 3: Identify the most significant gap between the candidate and the role's requirements.
- Sentence 4 (optional): One specific recommendation to strengthen the application.

Be direct. Keep to 3-4 sentences only.
"""
    return call_llm(prompt)

result_v3 = fit_summary_v3(target_jd, resume_text)
print("=== FIT SUMMARY v3: Goldman Sachs ===\n")
print(result_v3)

=== FIT SUMMARY v3: Goldman Sachs ===

**Fit Summary:** The candidate is a strong fit for the Analyst role in the XIG Data Office at Goldman Sachs due to their solid background in data analysis and experience with relevant tools. Their proficiency in SQL and Python, along with hands-on experience in developing KPI dashboards, aligns well with the responsibilities of monitoring data quality and supporting data governance initiatives. However, the candidate lacks direct experience in private markets or alternative investments, which is a preferred qualification for the role. To strengthen their application, they should emphasize any relevant coursework or projects related to asset management or financial services.


In [19]:
# ── Fit Summary: Iteration 3 ──
def fit_summary(jd_text, resume_text):
    prompt = f"""
You are a hiring consultant writing a brief fit assessment.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Write a 3-4 sentence narrative Fit Summary assessing how well this candidate fits the role.

- Sentence 1: State overall fit level (Strong / Moderate / Weak) and primary reason,
  citing specific evidence from both documents.
- Sentence 2: Highlight the candidate's single strongest relevant qualification for this role.
- Sentence 3: Identify the most significant gap between the candidate and the role's requirements.
- Sentence 4 (optional): One specific recommendation to strengthen the application.

Be direct and evidence-based. Do not use filler phrases.
Only reference information present in the documents provided.
"""
    return call_llm(prompt)

result_v4 = fit_summary(target_jd, resume_text)
print("=== FIT SUMMARY v4: Goldman Sachs ===\n")
print(result_v4)

=== FIT SUMMARY v4: Goldman Sachs ===

**Fit Summary:** The candidate is a **Strong** fit for the Analyst role in the XIG Data Office at Goldman Sachs due to their solid background in data analysis and experience with relevant tools such as SQL, Python, and Tableau, which align well with the job's technical requirements. Their current role as a Pricing Analyst, where they developed KPI dashboards and performed data extraction and transformation, demonstrates their ability to support data governance and quality initiatives effectively. However, the candidate lacks direct experience in private markets or alternative investments, which is preferred for this position. To strengthen their application, the candidate should emphasize any relevant coursework or projects related to asset management or private markets.


### Prompt Iteration Log

**Iteration 1 — Fit Summary (Base)**
Prompt: "Read this job description and resume and write a short summary of how
well the candidate fits the role."
Problem: The output was 4-5 paragraphs long, never stated an overall fit level,
and used generic language throughout with no evidence cited. The Goldman Sachs
output, for example, described the candidate as having "a proactive mindset" and
being "well-suited" for the role without pointing to anything specific in either
document. Impossible to compare across JDs.

**Iteration 2 — Fit Summary (Added fit level and length constraint)**
Change: Added "Begin with an overall fit level: Strong, Moderate, or Weak" and
capped the output at 3-4 sentences.
Improvement: Outputs became concise and always opened with a clear verdict. The
Goldman Sachs output came back as "Fit Level: Strong" followed by 3 sentences.
Remaining issue: The sentences were still vague. Phrases like "aligns well with
the responsibilities" appeared without citing what specifically aligned or which
JD requirement it matched.

**Iteration 3 — Fit Summary (Added sentence-level instructions)**
Change: Specified what each sentence had to cover: fit verdict with reason,
strongest qualification, biggest gap, and an optional recommendation.
Improvement: The output became consistently structured and directly comparable
across all three JDs. Each sentence had a clear job to do.
Remaining issue: Still no requirement to cite evidence, so claims like "solid
educational foundation relevant to the role" remained unsupported assertions.

**Iteration 4 — Fit Summary (Final: added evidence requirement)**
Change: Added "citing specific evidence from both documents" to sentence 1 and
"only reference information present in the documents provided" to the closing
instructions.
Improvement: Claims became verifiable. The Goldman Sachs output cited "developed
KPI dashboards in Tableau for a team of 5" matched to the JD requirement for
"preparing dashboards for internal stakeholders." Vague assertions like "solid
educational foundation" were replaced with specific evidence. This version was
used for all 9 evaluation analyses.

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [20]:
# ── Few-shot version of your chosen analysis ──
def skill_gap_report_fewshot(jd_text, resume_text):
    prompt = f"""
You are a career coach analyzing a candidate's resume against a job description.
Below is one example of the correct output format, followed by the actual task.

--- EXAMPLE ---
MATCHING SKILLS:
- SQL: Candidate lists SQL under Technical Skills and used it for data extraction at Onboard Logistics.
- Data Visualization: Candidate built Tableau dashboards for KPI reporting in current role.

MISSING SKILLS:
- dbt: JD requires dbt for pipeline management; not mentioned anywhere in the resume.
- A/B Testing: JD lists experimentation design as required; resume has no mention of this.

RECOMMENDED ACTIONS:
- dbt: Complete the free "dbt Fundamentals" course at courses.getdbt.com and build a personal project.
- A/B Testing: Take a Coursera or DataCamp course on A/B testing and apply it to a Kaggle dataset.
--- END EXAMPLE ---

Now perform the same analysis for the following:

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

MATCHING SKILLS:
MISSING SKILLS:
RECOMMENDED ACTIONS:
"""
    return call_llm(prompt)

In [21]:
# ── Run both on the same JD, display side by side ──
comparison_jd = get_jd_text("jd_09_bayview_data_ops_analyst.txt")

zero_shot_output = skill_gap_report(comparison_jd, resume_text)
few_shot_output  = skill_gap_report_fewshot(comparison_jd, resume_text)

print("=" * 60)
print("ZERO-SHOT OUTPUT — Bayview Asset Management")
print("=" * 60)
print(zero_shot_output)

print("\n" + "=" * 60)
print("FEW-SHOT OUTPUT — Bayview Asset Management")
print("=" * 60)
print(few_shot_output)

ZERO-SHOT OUTPUT — Bayview Asset Management
### SKILL GAP REPORT

#### MATCHING SKILLS:
- **SQL**: The candidate lists SQL as a technical skill, indicating proficiency in this required area.
- **Python**: The candidate has experience with Python, which is required for the position.
- **Data Analysis & Visualization**: The candidate is proficient in Tableau and Power BI, both of which are mentioned as desirable skills in the job description.
- **Data Extraction & Transformation**: The candidate has performed data extraction and transformation in their role as a Pricing Analyst, aligning with the job's requirement for ETL processes.
- **Analytical Skills**: The candidate has demonstrated strong analytical skills through their work experience and projects, such as analyzing freight rates and conducting a gender pay inequity analysis.
- **Communication Skills**: The candidate has experience delivering lessons and improving learning outcomes, showcasing their ability to communicate complex 

### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**
Skill Gap Report

**Which performed better?**
Few-shot performed better.

**Why? (use specific examples from the outputs above)**
I ran both versions on the Bayview Asset Management job description using the
same resume as input. The zero-shot prompt gave the model instructions and asked
it to complete the task. The few-shot prompt did the same thing but also included one example of what a correct output looks like before asking the model to produce its own.

The zero-shot version ignored the requested plain bullet format and added extra things throughout the output, such as bolding skill names and adding "####" section headers. The few-shot version followed the format in the example, producing clean plain text bullets with no extra styling. For a production application where the output gets rendered in a UI, extra markdown would either display incorrectly or require an extra parsing step to strip out.

Both versions identified the same four core missing skills: ETL tools, data
warehousing, cloud technologies, and financial services experience. The main
content difference was evidence quality. The zero-shot version wrote "has
experience with data extraction and transformation" while the few-shot version
wrote "has experience with data extraction and transformation at Onboard
Logistics." That employer-level citation makes the output verifiable.

The few-shot version also listed four missing skills compared to six in the
zero-shot version. The two extras in zero-shot (production support for ETL jobs
and collateral management) were legitimate gaps but had no resume evidence at all, making them weaker inclusions. The few-shot version was more conservative and better calibrated to what was actually in the documents.

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [22]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──
top_3_jds = [
    ("jd_10_goldmansachs_data_office_analyst.txt", "Goldman Sachs"),
    ("jd_09_bayview_data_ops_analyst.txt",          "Bayview Asset Management"),
    ("jd_04_delta_data_analyst.txt",                "Delta Air Lines"),
]

results_store = {}

for filename, company in top_3_jds:
    jd_text = get_jd_text(filename)
    results_store[company] = {
        "skill_gap":         skill_gap_report(jd_text, resume_text),
        "keyword_alignment": keyword_alignment(jd_text, resume_text),
        "fit_summary":       fit_summary(jd_text, resume_text)
    }
    print(f"Completed: {company}")

print("\nAll 9 analyses complete.")

Completed: Goldman Sachs
Completed: Bayview Asset Management
Completed: Delta Air Lines

All 9 analyses complete.


In [23]:
# ── Summarize evaluation results ──
for company, analyses in results_store.items():
    print("=" * 70)
    print(f"COMPANY: {company}")
    print("=" * 70)
    for analysis_type, output in analyses.items():
        print(f"\n--- {analysis_type.upper().replace('_', ' ')} ---")
        print(output)
    print()

COMPANY: Goldman Sachs

--- SKILL GAP ---
### SKILL GAP REPORT

#### MATCHING SKILLS:
- **Data Analysis Experience**: The candidate has experience as a Pricing Analyst where they performed data extraction and transformation, which aligns with the requirement for data analysis and operations experience.
- **Technical Skills**: Proficiency in SQL and Python is highlighted in the resume, matching the job's requirement for these programming languages.
- **Data Visualization**: The candidate developed KPI dashboards in Tableau, which corresponds to the need for preparing dashboards and reports for internal stakeholders.
- **Attention to Detail**: The candidate's work in improving win rates and generating gross profit demonstrates strong analytical skills and attention to detail.
- **Stakeholder Engagement**: Experience as a Professional Learning Facilitator involved delivering lessons and using performance data, indicating skills in communication and stakeholder management.
- **Interest in 

In [26]:
# ── Evaluation summary table ──
eval_data = {
    "Company": [
        "Goldman Sachs", "Goldman Sachs",    "Goldman Sachs",
        "Bayview",       "Bayview",          "Bayview",
        "Delta",         "Delta",            "Delta"
    ],
    "Analysis Type": [
        "Skill Gap", "Keyword Alignment", "Fit Summary",
        "Skill Gap", "Keyword Alignment", "Fit Summary",
        "Skill Gap", "Keyword Alignment", "Fit Summary"
    ],
    "Retrieval Relevance": [
        "Yes", "Yes", "Yes",
        "Yes", "Yes", "Yes",
        "Yes", "Yes", "Yes"
    ],
    "Skill ID Accuracy": [
        "5/6 correct", "11/15 correct", "N/A",
        "7/7 correct", "10/15 correct", "N/A",
        "5/5 correct", "11/15 correct", "N/A"
    ],
    "Actionability (1-5)": [
        4, 3, 4,
        4, 3, 4,
        4, 3, 4
    ],
    "Faithfulness": [
        "Partial", "Faithful", "Partial",
        "Faithful", "Partial", "Partial",
        "Faithful", "Faithful", "Faithful"
    ]
}

eval_df = pd.DataFrame(eval_data)
display(eval_df)

,Company,Analysis Type,Retrieval Relevance,Skill ID Accuracy,Actionability (1-5),Faithfulness
0,Goldman Sachs,Skill Gap,Yes,5/6 correct,4,Partial
1,Goldman Sachs,Keyword Alignment,Yes,11/15 correct,3,Faithful
2,Goldman Sachs,Fit Summary,Yes,N/A,4,Partial
3,Bayview,Skill Gap,Yes,7/7 correct,4,Faithful
4,Bayview,Keyword Alignment,Yes,10/15 correct,3,Partial
5,Bayview,Fit Summary,Yes,N/A,4,Partial
6,Delta,Skill Gap,Yes,5/5 correct,4,Faithful
7,Delta,Keyword Alignment,Yes,11/15 correct,3,Faithful
8,Delta,Fit Summary,Yes,N/A,4,Faithful


### Evaluation Analysis

**Which analysis type worked best?**
The Skill Gap Report worked best. The three-section structure made outputs easy
to verify manually, and the recommended actions were specific enough, naming real certifications, platforms, and tools rather than generic advice. It scored a 4 out of 5 on actionability across all three companies.

Keyword Alignment was the easiest to verify since every claim is a discrete table row I could check directly. It scored lower on actionability (3 out of 5) because it identifies gaps but does not recommend what to do about them. Fit Summary was readable and well structured after the final prompt iteration but harder to score for accuracy since it is a narrative rather than a checklist.

**Which JDs produced the best/worst results? Why?**
Bayview produced the strongest results because the JD was specific. It named
actual ETL tools, required data warehouse design explicitly, and listed cloud
technologies as a hard requirement. That gave the model clear criteria to compare against the resume, and 7 out of 8 matching skills held up when I verified them manually.

Goldman Sachs produced the weakest results. The JD mixes hard technical
requirements with soft preferences like "interest in private markets" and
"curiosity about AI" which are hard to evaluate from a resume. The model
struggled here and made loose inferences. It listed "Stakeholder Engagement" as a match based on the teaching role, then the keyword alignment output flagged the same skill as missing. Same data, opposite conclusions from two different prompts.

Delta fell in the middle. The candidate has real skill overlap with what Delta
needs, and the keyword alignment table correctly identified logistics experience
as partially relevant to Delta's transportation industry preference. The main
weakness was the fit summary calling the candidate a "strong fit" despite no
aviation-specific experience.

**Where did the system hallucinate or produce inaccurate results?**
The clearest error was in the Bayview Keyword Alignment output which marked
"Communication skills: MISSING" despite the skill being explicitly listed in the
resume and the candidate's entire teaching role demonstrating it. The model
searched for the exact string rather than matching semantically.

The Goldman Sachs Skill Gap report listed "Project Support" as a match based on
the candidate tracking timelines as a teaching facilitator. That is too loose a
connection given the role is about supporting data projects at a financial firm.

The Bayview Fit Summary also called the candidate a "Strong" fit while the
keyword alignment analysis on the same JD produced a 46.67% match rate. Those
two outputs directly contradict each other.

**What would you improve?**
The biggest improvement would be filtering retrieval by JD filename so only
chunks from the target document are returned. Right now all 10 JDs sit in the
same ChromaDB collection, which creates a risk of pulling content from the wrong
JD during retrieval, especially for similar roles like Goldman Sachs and Bayview.

I would also update the keyword alignment prompt to check for semantic equivalents before marking a skill as missing, which would have caught the communication skills error. And for the fit summary, I would pass in the keyword match rate as context so the fit verdict is grounded in a quantitative score rather than the model making its own holistic judgment.

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*